In [4]:
# ================================
# Movie Recommendation System
# ================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ast
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported! 🚀")

Libraries imported! 🚀


In [7]:
movies = pd.read_csv(r'D:\Shreyash_Projects\Movie-Recommendation\tmdb_5000_movies.csv')
credits = pd.read_csv(r'D:\Shreyash_Projects\Movie-Recommendation\tmdb_5000_credits.csv')

print(f"Movies shape: {movies.shape}")
print(f"Credits shape: {credits.shape}")
print(f"\nFirst movie: {movies['title'].iloc[0]}")

Movies shape: (4803, 20)
Credits shape: (4803, 4)

First movie: Avatar


In [8]:
# Check columns
print("Movies columns:")
print(movies.columns.tolist())

print("\nCredits columns:")
print(credits.columns.tolist())

print("\nMovies head:")
print(movies[['title', 'genres', 'vote_average', 'popularity']].head())

Movies columns:
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']

Credits columns:
['movie_id', 'title', 'cast', 'crew']

Movies head:
                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                              genres  vote_average  popularity  
0  [{"id": 28, "name": "Action"}, {"id": 12, "nam...           7.2  150.437577  
1  [{"id": 12, "name": "Adventure"}, {"id": 14, "...           6.9  139.082615  
2  [{"id": 28, "name": "Action"}, {"id": 12, "nam...           6.3  107.376788  
3  [{"id": 28, "name": "A

In [9]:
# Merge movies and credits
movies = movies.merge(credits, on='title')

print(f"After merge shape: {movies.shape}")

# Keep only useful columns
movies = movies[['id', 'title', 'overview', 'genres', 
                  'keywords', 'cast', 'crew', 
                  'vote_average', 'popularity', 'release_date']]

print(f"\nShape after selecting columns: {movies.shape}")
print(f"\nMissing values:\n{movies.isnull().sum()}")

After merge shape: (4809, 23)

Shape after selecting columns: (4809, 10)

Missing values:
id              0
title           0
overview        3
genres          0
keywords        0
cast            0
crew            0
vote_average    0
popularity      0
release_date    1
dtype: int64


In [10]:
# Function to extract names from JSON
def extract_names(text, top_n=None):
    try:
        items = ast.literal_eval(text)
        names = [item['name'] for item in items]
        if top_n:
            return names[:top_n]
        return names
    except:
        return []

# Extract director from crew
def extract_director(text):
    try:
        crew = ast.literal_eval(text)
        for member in crew:
            if member['job'] == 'Director':
                return [member['name'].replace(" ", "")]
        return []
    except:
        return []

# Apply extractions
movies['genres'] = movies['genres'].apply(extract_names)
movies['keywords'] = movies['keywords'].apply(extract_names)
movies['cast'] = movies['cast'].apply(lambda x: extract_names(x, top_n=3))
movies['crew'] = movies['crew'].apply(extract_director)

# Remove spaces from names
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])

print(movies[['title', 'genres', 'cast', 'crew']].head())

                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                         genres  \
0  [Action, Adventure, Fantasy, ScienceFiction]   
1                  [Adventure, Fantasy, Action]   
2                    [Action, Adventure, Crime]   
3              [Action, Crime, Drama, Thriller]   
4           [Action, Adventure, ScienceFiction]   

                                            cast                crew  
0  [SamWorthington, ZoeSaldana, SigourneyWeaver]      [JamesCameron]  
1     [JohnnyDepp, OrlandoBloom, KeiraKnightley]     [GoreVerbinski]  
2      [DanielCraig, ChristophWaltz, LéaSeydoux]         [SamMendes]  
3      [ChristianBale, MichaelCaine, GaryOldman]  [ChristopherNolan]  
4    [TaylorKitsch, LynnCollins, SamanthaMorton]     [A

In [11]:
# Fill missing overviews
movies['overview'] = movies['overview'].fillna('')

# Create tags — combine all features
movies['tags'] = (movies['overview'].apply(lambda x: x.split()) +
                  movies['genres'] +
                  movies['keywords'] +
                  movies['cast'] +
                  movies['crew'])

# Convert list to string
movies['tags'] = movies['tags'].apply(lambda x: ' '.join(x).lower())

# Keep final dataframe
df_final = movies[['id', 'title', 'tags', 'vote_average', 'popularity', 'release_date']].copy()

print(f"Final shape: {df_final.shape}")
print(f"\nSample tags for Avatar:")
print(df_final['tags'].iloc[0][:200])

Final shape: (4809, 6)

Sample tags for Avatar:
in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy


In [12]:
# TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
vectors = tfidf.fit_transform(df_final['tags'])

print(f"Vector shape: {vectors.shape}")

# Cosine Similarity matrix
similarity = cosine_similarity(vectors)

print(f"Similarity matrix shape: {similarity.shape}")
print(f"\nSimilarity between Avatar and Movie 2: {similarity[0][1]:.3f}")
print(f"Similarity between Avatar and Movie 3: {similarity[0][2]:.3f}")

Vector shape: (4809, 5000)
Similarity matrix shape: (4809, 4809)

Similarity between Avatar and Movie 2: 0.023
Similarity between Avatar and Movie 3: 0.013


In [13]:
# Recommendation function
def recommend(movie_title, n=10):
    # Find movie index
    movie_title_lower = movie_title.lower()
    matches = df_final[df_final['title'].str.lower().str.contains(movie_title_lower)]
    
    if matches.empty:
        return f"Movie '{movie_title}' not found!"
    
    idx = matches.index[0]
    movie_name = df_final.loc[idx, 'title']
    
    # Get similarity scores
    sim_scores = list(enumerate(similarity[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]  # exclude the movie itself
    
    # Get movie details
    recommendations = []
    for i, score in sim_scores:
        recommendations.append({
            'title': df_final.loc[i, 'title'],
            'similarity': round(score*100, 1),
            'rating': df_final.loc[i, 'vote_average'],
            'popularity': round(df_final.loc[i, 'popularity'], 1)
        })
    
    rec_df = pd.DataFrame(recommendations)
    print(f"\nTop {n} recommendations for '{movie_name}':\n")
    print(rec_df.to_string(index=False))
    return rec_df

# Test it!
recommend("Avatar")


Top 10 recommendations for 'Avatar':

                  title  similarity  rating  popularity
          Falcon Rising        20.9     5.5         7.0
    Battle: Los Angeles        19.9     5.5        49.2
              Apollo 18        18.8     5.0        17.0
Star Trek Into Darkness        17.5     7.4        78.3
             Titan A.E.        16.9     6.3        14.4
       The Book of Life        16.7     7.3        34.9
                 Aliens        16.7     7.7        67.7
           Ender's Game        16.4     6.6        45.9
                Jarhead        15.9     6.6        32.2
              Lifeforce        15.8     6.2        12.2


,title,similarity,rating,popularity
0,Falcon Rising,20.9,5.5,7.0
1,Battle: Los Angeles,19.9,5.5,49.2
2,Apollo 18,18.8,5.0,17.0
3,Star Trek Into Darkness,17.5,7.4,78.3
4,Titan A.E.,16.9,6.3,14.4
5,The Book of Life,16.7,7.3,34.9
6,Aliens,16.7,7.7,67.7
7,Ender's Game,16.4,6.6,45.9
8,Jarhead,15.9,6.6,32.2
9,Lifeforce,15.8,6.2,12.2


In [14]:
# Test with different movies
print("=" * 50)
recommend("The Dark Knight")
print("\n" + "=" * 50)
recommend("Avengers")


Top 10 recommendations for 'The Dark Knight Rises':

                                  title  similarity  rating  popularity
                        The Dark Knight        45.6     8.2       187.3
                         Batman Returns        40.0     6.6        59.1
                          Batman Begins        35.4     7.5       115.0
                         Batman Forever        33.3     5.2        48.2
                                 Batman        33.1     7.0        44.1
                                 Batman        30.2     7.0        44.1
                         Batman & Robin        26.2     4.2        50.1
Batman: The Dark Knight Returns, Part 2        24.6     7.9        25.9
     Batman v Superman: Dawn of Justice        24.3     5.7       155.8
                              Slow Burn        19.3     5.5         6.6


Top 10 recommendations for 'Avengers: Age of Ultron':

                     title  similarity  rating  popularity
                  Iron Man        32.9

,title,similarity,rating,popularity
0,Iron Man,32.9,7.4,120.7
1,Iron Man 3,31.6,6.8,77.7
2,The Avengers,31.5,7.4,144.4
3,Iron Man 2,30.1,6.6,77.3
4,Captain America: Civil War,26.0,7.1,198.4
5,Thor,22.7,6.6,86.5
6,Thor: The Dark World,21.8,6.8,99.5
7,Ant-Man,17.5,7.0,120.1
8,The Incredible Hulk,17.0,6.1,62.9
9,Superman II,16.8,6.5,30.5


In [15]:
import pickle
import os

# Create folder
os.makedirs('D:\\Shreyash_Projects\\Movie-Recommendation', exist_ok=True)

# Save similarity matrix and dataframe
with open('D:\\Shreyash_Projects\\Movie-Recommendation\\similarity.pkl', 'wb') as f:
    pickle.dump(similarity, f)

with open('D:\\Shreyash_Projects\\Movie-Recommendation\\movies.pkl', 'wb') as f:
    pickle.dump(df_final, f)

print("Model saved! ✅")

Model saved! ✅
